In [2]:
import pandas as pd
import numpy as np

url = (
    "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"
    "station=UNV&data=all&tz=America/New_York&format=onlycomma"
    "&latlon=yes&elev=yes&missing=M&trace=T"
    "&year1=2024&month1=4&day1=15&year2=2024&month2=4&day2=30"
)

df = pd.read_csv(url, na_values=["M", ""], parse_dates=["valid"], low_memory=False)
df["p01i"] = pd.to_numeric(df["p01i"].replace("T", 0.0001), errors="coerce")

# Derived fields
df["wind_speed_ms"] = df["sknt"] * 0.514444
df["tmpc"] = (df["tmpf"] - 32) * 5 / 9

# Max sky cover across layers
sky_rank = {"CLR": 0, "FEW": 2, "SCT": 4, "BKN": 6, "OVC": 8, "VV": 8}
df["sky_oktas"] = df[["skyc1","skyc2","skyc3"]].apply(
    lambda r: max((sky_rank.get(v, -1) for v in r if pd.notna(v)), default=np.nan), axis=1
)

df.to_csv("UNV_ASOS_apr_2024.csv", index=False)
print(f"{len(df)} observations saved")

599 observations saved


In [3]:
"""
Download ASOS weather data from Iowa Environmental Mesonet
for BeeMonitor foraging analysis.

Run this script locally to download weather data.

Station: UNV (University Park, PA - State College)
Period: April 1 - June 30, 2024
"""

import pandas as pd
import numpy as np

# IEM ASOS download URL
url = (
    "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"
    "station=UNV&data=all&tz=America/New_York&format=onlycomma"
    "&latlon=yes&elev=yes&missing=M&trace=T"
    "&year1=2024&month1=4&day1=1&year2=2024&month2=6&day2=30"
)

print("Downloading weather data from IEM...")
df = pd.read_csv(url, na_values=["M", ""], parse_dates=["valid"], low_memory=False)
print(f"Downloaded {len(df)} observations")

# Convert trace precipitation to small value
df["p01i"] = pd.to_numeric(df["p01i"].replace("T", 0.0001), errors="coerce")

# Derived fields
df["wind_speed_ms"] = df["sknt"] * 0.514444
df["tmpc"] = (df["tmpf"] - 32) * 5 / 9

# Max sky cover across layers
sky_rank = {"CLR": 0, "FEW": 2, "SCT": 4, "BKN": 6, "OVC": 8, "VV": 8}
df["sky_oktas"] = df[["skyc1", "skyc2", "skyc3"]].apply(
    lambda r: max((sky_rank.get(v, -1) for v in r if pd.notna(v)), default=np.nan), axis=1
)

# Save full hourly data
df.to_csv("UNV_ASOS_apr_jun_2024.csv", index=False)
print(f"Saved: UNV_ASOS_apr_jun_2024.csv ({len(df)} rows)")

# Create daily summary
df["date"] = df["valid"].dt.date
daily = df.groupby("date").agg({
    "tmpc": ["min", "max", "mean"],
    "p01i": "sum",
    "wind_speed_ms": "mean",
    "relh": "mean",
    "sky_oktas": "mean"
}).reset_index()
daily.columns = ["date", "temp_min", "temp_max", "temp_mean", "precip_in", "wind_ms", "relh", "sky_oktas"]
daily["precip_mm"] = daily["precip_in"] * 25.4

daily.to_csv("UNV_daily_apr_jun_2024.csv", index=False)
print(f"Saved: UNV_daily_apr_jun_2024.csv ({len(daily)} days)")

print(f"\nDate range: {daily['date'].min()} to {daily['date'].max()}")

Downloaded 3227 observations
Saved: UNV_ASOS_apr_jun_2024.csv (3227 rows)
Saved: UNV_daily_apr_jun_2024.csv (90 days)

Date range: 2024-04-01 to 2024-06-29
